In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import GDLModules.util as util
import torch.nn.functional as F
import plotly.express as px
import torch.nn as nn
import torch
import GDLModules.A_CAE as A_CAE
import GDLModules.A_AE as A_AE
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.datasets import make_s_curve
from geomloss import SamplesLoss
import copy
from tqdm import tqdm, trange
from torch.utils.data import DataLoader

d:\Users\kevin\miniconda3\envs\test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [272]:
torus_data = util.generate_parabola_surface(10000, depth=4)
torus_data_test = util.generate_parabola_surface(10000)
util.plot3d([(torus_data, "torus data")], title="Double Torus")

In [273]:
class AE(nn.Module):
    def __init__(self, x_dim, z_dim, hidden_structure):
        super().__init__()
        self.center = nn.Parameter(torch.zeros(x_dim), requires_grad=True)
        self.x_dim = x_dim
        self.z_dim = z_dim
        self.encoder = util.Sum(
            nn.Linear(x_dim, z_dim, bias=False),
            util.MLP(x_dim, z_dim, hidden_structure, activation=nn.Tanh()),
        )
        self.decoder = util.Sum(
            nn.Linear(z_dim, x_dim, bias=False),
            util.MLP(z_dim, x_dim, hidden_structure, activation=nn.Tanh()),
        )





    def encode(self, x):
        z = self.encoder(x-self.center)
        return z

    def decode(self, z):
        x = self.decoder(z)
        return x + self.center


    def encode_single(self, x):
        return self.encode(x.unsqueeze(0)).squeeze(0)

    def decode_single(self, x):
            return self.decode(x.unsqueeze(0)).squeeze(0)


    def encode_decode(self, x):
        z_latent = self.encode(x)
        x_recon = self.decode(z_latent)
        return x_recon

In [274]:
criterion = SamplesLoss(
        loss="sinkhorn", p=2, blur=0.05, scaling=0.8, backend="tensorized", debias=True
    )


def train_ae_sinkhorn(num_epochs: int, x_train, network: AE, device: torch.device, lr = 0.001):
    network.to(device)
    optimizer = torch.optim.Adam(network.parameters(), lr=lr)
    pbar = trange(num_epochs)
    x_train = x_train.to(device)
    x_train_loader = DataLoader(x_train, batch_size=512, shuffle=True)
    for epoch in pbar:

        for batch in x_train_loader:
            # sample 1000 points
            optimizer.zero_grad()
            z_sample = torch.rand(512, network.z_dim).to(device)
            x_decoded = network.decode(z_sample)
            z_ecoded = network.encode(x_decoded)
            loss = criterion(x_decoded, batch) + F.mse_loss(z_ecoded, z_sample)
            pbar.set_postfix({"loss": loss.item()})
            loss.backward()
            optimizer.step()
    return





In [275]:
ae = AE(x_dim=3, z_dim=2, hidden_structure=[64, 64, 64])
ae = ae.to(device)
ae = ae.train()

In [277]:

train_ae_sinkhorn(num_epochs=200, x_train=torch.tensor(torus_data, dtype=torch.float32).to(device), network=ae, device=device, lr=1e-3)

train_ae_sinkhorn(num_epochs=200, x_train=torch.tensor(torus_data, dtype=torch.float32).to(device), network=ae, device=device, lr=1e-5)

 44%|████▍     | 89/200 [01:18<01:38,  1.13it/s, loss=0.126]


KeyboardInterrupt: 

In [ ]:
torch.save(ae.state_dict(), "ae_quater_torus_sinkhorn2.pt")

In [278]:
z_sample = torch.rand(1000, ae.z_dim).to(device)
x_sample_t = ae.decode(z_sample)
x_sample = x_sample_t.detach().cpu().numpy()
util.plot3d([(x_sample, "decoded torus"), (torus_data[0:1000], "torus data")], title="Decoded Torus from Latent Space")

# Single Chart Geodesics

In [ ]:
point_a = np.array([0.2, 0.2])
point_b = np.array([0.1, 0.8])



#linear interpolation
num_points = 50
interpolated_points = torch.tensor(np.linspace(point_a, point_b, num=num_points), device=device, dtype=torch.float)

len(interpolated_points)

decoded_interpolated_points_t = ae.decode(torch.tensor(interpolated_points, dtype=torch.float32).to(device))
decoded_interpolated_points = decoded_interpolated_points_t.detach().cpu().numpy()
# add decoded points to x_decoded
x_sample_e = np.concatenate([x_sample, decoded_interpolated_points[[0, -1]]], axis=0)


C:\Users\kevin\AppData\Local\Temp\ipykernel_13896\2763421676.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  decoded_interpolated_points_t = ae.decode(torch.tensor(interpolated_points, dtype=torch.float32).to(device))


In [ ]:

util.plot3d([(x_sample_e, "decoded torus"), (decoded_interpolated_points, "interpolated_points", "lines", 10)], title="Decoded Torus from Latent Space")

In [ ]:
import time
from torch.func import jacfwd
z_dim = 2
lr = 0.005


geodesic = interpolated_points.clone()

def single_jac(x):
    return jacfwd(ae.encode_single)(x)


for j in trange (1500):
    pointwise_grads = torch.zeros(num_points, z_dim, device=device)
    idx = torch.arange(num_points, device=device)
    decoded_geodesic = ae.decode(geodesic)

    J = torch.vmap(single_jac)(decoded_geodesic)

    for i in range(1, num_points-1):
        pointwise_grads[i, :] = -num_points * J[i,:,:] @ (decoded_geodesic[i+1] - 2*decoded_geodesic[i] + decoded_geodesic[i-1])
    geodesic.device
    geodesic -= lr * pointwise_grads




100%|██████████| 1500/1500 [00:20<00:00, 72.40it/s]


In [ ]:
from sklearn.neighbors import kneighbors_graph
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
sparse_graph  = kneighbors_graph(x_sample_e, 10, mode='distance', include_self=False)
distances, predecessors = dijkstra(
    csgraph=sparse_graph, 
    directed=False, 
    indices=1000, 
    return_predecessors=True
)
path = []
curr = 1001
while curr != -9999:
    path.append(curr)
    curr = predecessors[curr]
path.reverse()
path = np.array(path)

In [ ]:
decoded_geodesic = ae.decode(geodesic).detach().cpu().numpy()
util.plot3d([(x_sample, "decoded torus"), (decoded_interpolated_points, "interpolated_points", "lines", 10), (decoded_geodesic, "decoded_geodesic", "lines", 10), (x_sample_e[path], "dijkstra", "lines", 10)], title="Decoded Torus from Latent Space")